# Fase 6: Modelado de temas
---
Este cuaderno realiza un *benchmark* evolutivo, comenzando con algoritmos estadísticos clásicos como LDA o NMF y culminando con arquitecturas basadas en Transformers como BERTopic. También se integra el *Large Language Model* Llama-3 para traducir las representaciones vectoriales matemáticas en nombres de categorías legibles para el usuario final.

A diferencia de la clasificación Zero-Shot (que asigna etiquetas predefinidas), el *topic modeling* es una técnica de aprendizaje no supervisado. Su objetivo es descubrir estructuras ocultas y agrupaciones semánticas orgánicas dentro de la discografía, sin intervención humana previa.

In [2]:
import pandas as pd
import os

from groq import Groq
from dotenv import load_dotenv

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

import warnings
warnings.filterwarnings('ignore')

In [3]:
load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [4]:
df = pd.read_csv('../data/processed/taylor_swift_metrics.csv')
lyrics = df['lyrics_unique'].dropna().tolist()

### 6.1 Eliminación de ruido
Los modelos de *topic modeling* son extremadamente sensibles al ruido léxico. Si no se aplican filtros, el algoritmo agrupará las canciones basándose en palabras que se repiten mucho en las canciones (como "oh" o "yeah") en lugar de en su carga semántica real. Por ello, se han extendido las *stopwords* nativas del inglés con una lista personalizada, garantizando que los clústeres se formen en torno a sustantivos y verbos con peso narrativo.

In [5]:
custom_stopwords = list(ENGLISH_STOP_WORDS) + [
    'oh', 'yeah', 'cause', 'wanna', 'gonna', 'gotta', 'like', 'just', 'got', 'woah',
    'know', 'don', 'll', 've', 're', 'ain', 'ooh', 'ah', 'ha', 'la', 'eh', 'na', 'til',
    'im', 'm', 'did', 'didn', 'say', 'said', 'let', 'tell', 'way', 'make', 'wouldve',
    'uh', 'want', 'need', 'feel', 'think', 'right', 'time', 'thing', 'look', 'come',
    'dont', 'youre', 'ill', 'ive', 'hes', 'shes', 'were', 'theres', 'cant', 'wont',
    'didnt', 'id', 'youd', 'shouldve', 'isnt', 'youve', 'arent', 'wasnt', 'werent',
    'hadnt', 'doesnt', 'couldve', 'theyre', 'hey', 'thats', 'hasnt', 'isn', 'mm', 'won',
    'wasn', 'weren', 'doesn', 'hasn', 'hadn', 'couldn', 'wouldn', 'shouldn', 'aren',
    'somethin', 'nothin', 'bout', 'em', 'things', 'waitin', 'youll', 'theyll', 'ohohoh', 'oohayy'
]

### 6.2 Latent Dirichlet Allocation (LDA)
Creado en 2003, es el algoritmo estadístico clásico por excelencia. Asume que cada documento es una mezcla de temas y cada tema es una mezcla de palabras.

- *Pros*: muy rápido, funciona bien con textos largos como libros o noticias.
- *Contras*: no suele funcionar igual de bien en textos cortos y poéticos, no entiende el contexto de las palabras (sólo cuenta cuántas veces aparecen juntas).

In [6]:
print("Entrenando modelo LDA...")

vectorizer = CountVectorizer(stop_words=custom_stopwords, max_features=1000)
X = vectorizer.fit_transform(lyrics)

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

print("\nResultados LDA:")
lda_words = vectorizer.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top_words = [lda_words[i] for i in topic.argsort()[-10:][::-1]]
    print(f"Topic {i+1}: {', '.join(top_words)}")

Entrenando modelo LDA...

Resultados LDA:
Topic 1: love, good, home, red, face, babe, away, eyes, girl, head
Topic 2: man, life, wish, remember, love, hold, lost, mind, gave, better
Topic 3: love, baby, night, girl, away, good, new, eyes, day, forever
Topic 4: baby, break, love, talk, knew, better, friends, remember, away, night
Topic 5: love, baby, bad, night, stay, blood, hands, hope, home, away


### 6.3 Non-Negative Matrix Factorization (NMF)
Modelo de álgebra lineal que factoriza la matriz de palabras.

- *Pros*: suele sacar temas más coherentes y específicos que LDA en textos cortos.
- *Contras*: no entiende la semántica.

In [7]:
print("\nEntrenando modelo NMF...")

vectorizer_nmf = TfidfVectorizer(stop_words=custom_stopwords, max_features=1000)
X_nmf = vectorizer_nmf.fit_transform(lyrics)

nmf = NMF(n_components=6, random_state=42)
nmf.fit(X_nmf)

print("\nResultados NMF:")
nmf_words = vectorizer_nmf.get_feature_names_out()
for i, topic in enumerate(nmf.components_):
    top_words = [nmf_words[i] for i in topic.argsort()[-10:][::-1]]
    print(f"Topic {i+1}: {', '.join(top_words)}")


Entrenando modelo NMF...

Resultados NMF:
Topic 1: baby, knew, away, run, girl, nice, sorry, eyes, night, smile
Topic 2: love, life, good, thought, heart, saw, bad, game, mind, touch
Topic 3: drew, beautiful, break, bet, guitar, talks, breathe, girl, fall, reason
Topic 4: remember, hope, lost, maybe, red, old, long, little, hair, days
Topic 5: stay, hold, leave, talk, people, away, easy, forever, babe, mad
Topic 6: man, better, wish, sick, good, case, shake, push, night, talk


### 6.4 BERTopic
Estado del arte actual (2020+). Utiliza redes neuronales (Transformers) para "leer" las canciones, generar *embeddings* contextuales y agrupar los conceptos matemáticamente.

- *Pros*: entiende el sarcasmo, los sinónimos y la poesía mucho mejor que los modelos estadísticos, suele crear clústeres muy coherentes.
- *Contras*: más lento y pesado de ejecutar.

In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2") # modelo por defecto que usa BERTopic en inglés
embeddings = embedding_model.encode(lyrics, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [9]:
k_values = [6, 7, 8, 9, 10, 11, 12]

best_models = {}
results = []

print("\nEntrenando modelo BERTopic...")

vectorizer_model = CountVectorizer(stop_words=custom_stopwords)

for k in k_values:
    cluster_model = KMeans(n_clusters=k, random_state=42)

    topic_model = BERTopic(
        language="english", 
        vectorizer_model=vectorizer_model,
        hdbscan_model=cluster_model,
        top_n_words=50
    )

    topics, probs = topic_model.fit_transform(lyrics, embeddings)

    score = silhouette_score(embeddings, topics)
    print(f"K={k} | Silhouette Score: {round(score, 4)}")

    results.append({'K': int(k), 'Silhouette Score': score})
    best_models[k] = topic_model

df_results = pd.DataFrame(results)
best_k = df_results.loc[df_results['Silhouette Score'].idxmax()]['K']

print("\n" + "="*50)
print(f"El número óptimo de clústeres es: K={int(best_k)}")
print("="*50)


Entrenando modelo BERTopic...
K=6 | Silhouette Score: 0.0057
K=7 | Silhouette Score: 0.0181
K=8 | Silhouette Score: 0.0148
K=9 | Silhouette Score: 0.0096
K=10 | Silhouette Score: 0.0124
K=11 | Silhouette Score: 0.0112
K=12 | Silhouette Score: 0.0089

El número óptimo de clústeres es: K=7


In [10]:
print("\nResultados BERTopic:")
info_topics = best_models[best_k].get_topic_info()

for idx, row in info_topics.iterrows():
    if row['Topic'] != -1:
        words_topic = best_models[best_k].get_topic(row['Topic'])
        top_words = [word[0] for word in words_topic[:50]]
        print(f"Topic {row['Topic']+1} ({row['Count']} canciones): {', '.join(top_words)}")


Resultados BERTopic:
Topic 1 (55 canciones): love, baby, away, stay, girl, run, wish, miss, talk, eyes, night, better, mr, forever, heart, babe, break, new, end, mind, shake, aint, fall, life, gettin, smile, touch, face, thought, standin, best, feelin, york, gets, leave, people, shine, man, try, places, jump, gone, trying, help, bad, comin, whyd, iii, hear, walk
Topic 2 (49 canciones): good, bad, daylight, karma, light, night, away, love, blood, heart, man, used, long, sky, life, single, losin, florida, day, lost, room, watch, really, war, fight, different, die, hate, friends, hell, people, hands, town, hits, screaming, wake, new, years, eyes, leave, mind, story, head, believed, house, tried, speak, run, old, words
Topic 3 (36 canciones): remember, dancin, lost, love, home, baby, woods, nice, hope, knew, night, forget, day, good, grow, little, old, away, twentytwo, used, hold, song, clear, trouble, long, left, gone, car, walked, dance, walk, lookin, feelin, fall, new, door, lose, frie

BERTopic utiliza por defecto un algoritmo de agrupación basado en densidad llamado HDBSCAN. Si una canción no encaja perfectamente en un grupo, este algoritmo la clasifica como basura/ruido (topic -1).

Como el conjunto de datos que se está utilizando está formado tan solo por 242 canciones, HDBSCAN no encuentra suficiente densidad de puntos.

Para mitigar este problema, el propio creador del modelo recomienda apagar HDBSCAN y sustituirlo por k-means cuando se tienen menos de 1000 documentos, por lo que se ha realizado dicho cambio, iterando sobre un rango de hiperparámetros `k` entre 6 y 12.

Además, se ha evaluado el *silhouette score* para determinar matemáticamente el número óptimo de clústeres.

In [11]:
winners = best_models[best_k].topics_
df['topic_bertopic'] = winners

In [12]:
print("Recopilando el panorama global de clústeres...\n")

global_context = ""
words_dict = {}

for idx, row in info_topics.iterrows():
    topic_id = row['Topic']
    
    if topic_id != -1:
        words_topic = best_models[best_k].get_topic(topic_id)
        top_words = ", ".join([word[0] for word in words_topic[:50]])
        words_dict[topic_id] = top_words
        global_context += f"- Cluster {topic_id}: {top_words}\n"

Recopilando el panorama global de clústeres...



In [ ]:
names = {}
words = words_dict
already_used_names = []

for topic_id, top_words in words_dict.items():
        used = ", ".join(already_used_names) if already_used_names else "Ninguno todavía"

        prompt = f"""
        You are a highly analytical music data scientist creating strict, conceptual taxonomy tags for Taylor Swift's discography.
        I have grouped her songs into several clusters. Here is the FULL LANDSCAPE of all clusters' words to prevent overlap:
        
        {global_context}

        TAGS ALREADY ASSIGNED TO OTHER CLUSTERS (DO NOT REPEAT THESE):
        [{used}]

        Keywords for cluster {topic_id}:
        [{top_words}]

        Your task: Categorize ONLY "Cluster {topic_id}" into a broad, recognizable "Song Theme" or "Musical Trope" in Spanish.

        CRITICAL RULES:
        1. The name MUST be a direct, conceptual category tag. Do NOT use poetic, dramatic, or storytelling phrases.
        - GOOD examples: "Nostalgia y madurez", "Amor tóxico", "Crítica a la fama", "Búsqueda de la esencia", "Romance idealizado",
        "Rebeldía juvenil", "Dolor y traición", "Himnos de ruptura", "Enamoramiento idealizado", "Traición y venganza". 
        You can get inspired but these examples, but do not copy them, invent your own based on the keywords.
        - BAD examples (DO NOT DO THIS): "En el umbral de la desesperanza", "Memorias de un amor en fuga", "Amor y libertad perdida",
        "Romance con sombras oscuras", "Recuerdos de la infancia".
        2. LENGTH: MAXIMUM 3-5 WORDS.
        3. OUTPUT FORMAT: ONLY output the exact category name in Spanish. No intros, no quotes, no explanations. 
        """

        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {"role": "system", "content": "You are a precise music data taxonomy assistant."},
                    {"role": "user", "content": prompt}
                ],
                model="llama-3.1-8b-instant",
                temperature=0.4,
                max_tokens=15
            )

            llm_name = chat_completion.choices[0].message.content.strip().replace('"', '')
            
            if ":" in llm_name:
                 llm_name = llm_name.split(":")[-1].strip()
            llm_name = llm_name.split('\n')[0].strip()
            
            names[topic_id] = llm_name
            already_used_names.append(llm_name)

            print(f"Topic {topic_id} -> {llm_name}\n (Palabras: {top_words})\n")
        
        except Exception as e:
            print(f"Error en el topic {topic_id}: e")
            names[topic_id] = f"Topic {topic_id}"

Topic 0 -> Amor y relaciones tumultuosas
 (Palabras: love, baby, away, stay, girl, run, wish, miss, talk, eyes, night, better, mr, forever, heart, babe, break, new, end, mind, shake, aint, fall, life, gettin, smile, touch, face, thought, standin, best, feelin, york, gets, leave, people, shine, man, try, places, jump, gone, trying, help, bad, comin, whyd, iii, hear, walk)

Topic 1 -> Crítica a la sociedad.
 (Palabras: good, bad, daylight, karma, light, night, away, love, blood, heart, man, used, long, sky, life, single, losin, florida, day, lost, room, watch, really, war, fight, different, die, hate, friends, hell, people, hands, town, hits, screaming, wake, new, years, eyes, leave, mind, story, head, believed, house, tried, speak, run, old, words)

Topic 2 -> Nostalgia y madurez
 (Palabras: remember, dancin, lost, love, home, baby, woods, nice, hope, knew, night, forget, day, good, grow, little, old, away, twentytwo, used, hold, song, clear, trouble, long, left, gone, car, walked, danc

In [14]:
df['topic_name_llm'] = df['topic_bertopic'].map(names)
df['topic_words'] = df['topic_bertopic'].map(words)

In [15]:
df.to_csv('../data/processed/taylor_swift_metrics.csv', index=False)

print(df['topic_bertopic'].value_counts().sort_index())

topic_bertopic
0    55
1    49
2    36
3    34
4    27
5    25
6    16
Name: count, dtype: int64
